# FHIR Patient Bronze-to-Silver Transformation

## Purpose

Transform raw FHIR Patient resources from the Bronze layer into a
structured and analytics-ready Silver Delta table.

### Source
`health_insurance.bronze.fhir_patient_raw`

### Target
`health_insurance.silver.fhir_patient`

### Responsibilities

- Parse raw FHIR JSON
- Extract nested Patient attributes
- Handle FHIR arrays and structures
- Extract patient identifiers
- Standardize dates and categorical fields
- Parse address information
- Preserve source lineage metadata

Formal data-quality enforcement is implemented separately in the
dedicated data-quality stage.



In [0]:

# stage configuration


CATALOG = "health_insurance"

SOURCE_TABLE = f"{CATALOG}.bronze.fhir_patient_raw"
TARGET_TABLE = f"{CATALOG}.silver.fhir_patient"

print("Source:", SOURCE_TABLE)
print("Target:", TARGET_TABLE)

In [0]:

# Loading Bronze FHIR Patient data


patient_bronze_df = spark.table(SOURCE_TABLE)

print(f"Rows: {patient_bronze_df.count():,}")
print(f"Columns: {len(patient_bronze_df.columns)}")

patient_bronze_df.printSchema()

display(patient_bronze_df.limit(5))

## Infer FHIR Patient JSON Schema

Databricks Serverless uses Spark Connect and does not support Spark RDD APIs.

The raw Patient JSON schema is therefore inferred directly from the
Bronze `raw_json` column using `schema_of_json_agg`.

Using the aggregate form allows Databricks to combine the structures
found across all Patient records, which is useful because FHIR fields
can be optional and vary between resources.

In [0]:

# Infer combined FHIR Patient schema
# Serverless-compatible implementation


schema_result = spark.sql(f"""
    SELECT schema_of_json_agg(raw_json) AS patient_schema
    FROM {SOURCE_TABLE}
""").first()

patient_schema = schema_result["patient_schema"]

print("FHIR Patient schema:")
print(patient_schema)

In [0]:

# Parsing the raw FHIR Patient JSON


from pyspark.sql import functions as F

patient_parsed_df = (
    patient_bronze_df
    .withColumn(
        "patient",
        F.from_json(
            F.col("raw_json"),
            patient_schema
        )
    )
)

patient_parsed_df.printSchema()

In [0]:

# Inspecting parsed Patient resources


display(
    patient_parsed_df.select(
        F.col("patient.id").alias("patient_id"),
        F.col("patient.gender").alias("gender"),
        F.col("patient.birthDate").alias("birth_date"),
        F.col("patient.name").alias("names"),
        F.col("patient.address").alias("addresses"),
        F.col("patient.telecom").alias("telecom")
    ).limit(10)
)

In [0]:

# Extract core Patient attributes


patient_core_df = (
    patient_parsed_df
    .select(
        F.col("patient.id").alias("patient_id"),

        F.col("patient.gender").alias("gender"),

        F.to_date(
            F.col("patient.birthDate")
        ).alias("birth_date"),

        F.col("patient.name").alias("name"),

        F.col("patient.address").alias("address"),

        F.col("patient.telecom").alias("telecom"),

        F.col("patient.identifier").alias("identifier"),

        F.col("patient.communication").alias("communication"),

        F.col("patient.maritalStatus").alias("marital_status"),

        F.col("patient.deceasedDateTime").alias("deceased_datetime"),

        "_ingested_at",
        "_source_system",
        "_resource_type"
    )
)

In [0]:

# Extracting official Patient name


patient_name_df = (
    patient_core_df

    .withColumn(
        "official_names",
        F.expr(
            """
            filter(
                name,
                x -> x.use = 'official'
            )
            """
        )
    )

    .withColumn(
        "official_name",
        F.element_at(
            F.col("official_names"),
            1
        )
    )

    .withColumn(
        "given_name",
        F.element_at(
            F.col("official_name.given"),
            1
        )
    )

    .withColumn(
        "family_name",
        F.col("official_name.family")
    )
)

In [0]:

# Extracting primary Patient address


patient_address_df = (
    patient_name_df

    .withColumn(
        "primary_address",
        F.element_at(
            F.col("address"),
            1
        )
    )

    .withColumn(
        "address_line",
        F.element_at(
            F.col("primary_address.line"),
            1
        )
    )

    .withColumn(
        "city",
        F.col("primary_address.city")
    )

    .withColumn(
        "state",
        F.col("primary_address.state")
    )

    .withColumn(
        "postal_code",
        F.col("primary_address.postalCode")
    )

    .withColumn(
        "country",
        F.col("primary_address.country")
    )
)

In [0]:

# Extract Patient phone number


patient_contact_df = (
    patient_address_df

    .withColumn(
        "phone_contacts",
        F.expr(
            """
            filter(
                telecom,
                x -> x.system = 'phone'
            )
            """
        )
    )

    .withColumn(
        "phone",
        F.element_at(
            F.col("phone_contacts.value"),
            1
        )
    )
)

We deliberately will not put SSN, passport number or driver's licence into our normal Silver analytical Patient table.

That's useful later for our Unity Catalog security/governance section.

In [0]:

# Extract Medical Record Number


patient_identifier_df = (
    patient_contact_df

    .withColumn(
        "medical_record_identifiers",
        F.expr(
            """
            filter(
                identifier,
                x -> exists(
                    x.type.coding,
                    c -> c.code = 'MR'
                )
            )
            """
        )
    )

    .withColumn(
        "medical_record_number",
        F.element_at(
            F.col("medical_record_identifiers.value"),
            1
        )
    )
)

In [0]:

# Deriving Patient age


patient_enriched_df = (
    patient_identifier_df

    .withColumn(
        "age",
        F.floor(
            F.months_between(
                F.current_date(),
                F.col("birth_date")
            ) / 12
        ).cast("int")
    )
)

In [0]:

# Creating Patient age groups


patient_enriched_df = (
    patient_enriched_df

    .withColumn(
        "age_group",

        F.when(
            F.col("age") < 18,
            "UNDER_18"
        )

        .when(
            F.col("age") < 35,
            "18_34"
        )

        .when(
            F.col("age") < 50,
            "35_49"
        )

        .when(
            F.col("age") < 65,
            "50_64"
        )

        .otherwise("65_PLUS")
    )
)

In [0]:

# Standardizing Patient text attributes


patient_standardized_df = (
    patient_enriched_df

    .withColumn(
        "gender",
        F.upper(F.trim("gender"))
    )

    .withColumn(
        "city",
        F.initcap(F.trim("city"))
    )

    .withColumn(
        "state",
        F.upper(F.trim("state"))
    )

    .withColumn(
        "country",
        F.upper(F.trim("country"))
    )
)

In [0]:

# Building final Silver Patient dataset


patient_silver_df = (
    patient_standardized_df

    .select(
        "patient_id",
        "medical_record_number",

        "given_name",
        "family_name",

        "gender",
        "birth_date",
        "age",
        "age_group",

        "phone",

        "address_line",
        "city",
        "state",
        "postal_code",
        "country",

        "_source_system",
        "_resource_type",
        "_ingested_at"
    )

    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp()
    )
)

In [0]:

# Inspecting Silver Patient result


patient_silver_df.printSchema()

display(
    patient_silver_df.limit(20)
)

In [0]:

# Reconciling Bronze and Silver row counts


bronze_count = patient_bronze_df.count()
silver_count = patient_silver_df.count()

print(f"Bronze Patients: {bronze_count:,}")
print(f"Silver Patients: {silver_count:,}")
print(f"Difference: {bronze_count - silver_count:,}")

In [0]:

# Persisting to FHIR Patient Silver table


(
    patient_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(TARGET_TABLE)
)

print(
    f"Created Silver table: {TARGET_TABLE}"
)

In [0]:
%sql
-- verifying the number of patients in the silver table

SELECT COUNT(*) AS patient_count
FROM health_insurance.silver.fhir_patient;